#### Sample script for reading in TC-DROPS data
Brian Tang

<i> 6 November 2025 </i>

In [ ]:
#Libraries

import re
import numpy as np
import xarray as xr
import pandas as pd
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
from metpy.calc import dewpoint_from_relative_humidity
from metpy.plots import SkewT
from metpy.units import units
import seaborn as sns
sns.set_context("paper",font_scale=1.0)

%matplotlib inline

In [ ]:
#open TC-DROPS dataset (we'll open the data on height levels)
ds_tcdrops = xr.open_dataset('/network/rit/lab/tcdynasty/tang/tcdrops/all_storms_height_dropsonde_dataset_ver1.2_updated.nc')

#print dataset info
ds_tcdrops

In [ ]:
#Specify storm and mission(s) of interest
atcfid = 'AL062014' #ATCF identifier
tcdropsid = 'al_edouard2014' #TCDROPS identifier <basin>_<name><year>
storm = re.search(r'_(\D+)\d+', tcdropsid).group(1) #extract just the storm name

#print all available missions for storm
stormname = np.squeeze(ds_tcdrops['storm_name'].values).astype(str)
ind = np.nonzero(stormname == tcdropsid)[0]
flightids = np.squeeze(ds_tcdrops['flight_id'].values).astype(str)
print(np.unique(flightids[ind]))

#specify missions of interest that you want to extract dropsonde data for
flightid = ['20140914I1', '20140914_hs3_gh'] #can list one mission; group missions together to examine combined dropsondes

In [ ]:
#Open IBTracs to get TC track observations
ds_ibtracs = xr.open_dataset('/network/rit/lab/tcdynasty/tang/ibtracs/IBTrACS.since1980.v04r01.nc')

#determine index of TC
atcf = np.squeeze(ds_ibtracs['usa_atcf_id'][:,0].values).astype(str)
atcfind = np.nonzero(atcf == atcfid)[0][0]
print(atcfind)

#get track information of TC
ds_ibtracs_tc = ds_ibtracs[['usa_lat','usa_lon']].isel(storm = atcfind)

#convert to dataframe
df_ibtracs_tc = ds_ibtracs_tc.to_dataframe().dropna(how='all').reset_index()
df_ibtracs_tc = df_ibtracs_tc.drop(columns=['date_time','usa_lat','usa_lon'])
df_ibtracs_tc['time'] = df_ibtracs_tc['time'].dt.round('min')

#resample to every minute and apply cubic interpolation
df_ibtracs_tc = df_ibtracs_tc.set_index('time').resample('1s').interpolate(method='cubic')
df_ibtracs_tc = df_ibtracs_tc.reset_index()

# calculate storm motion
REarth = 6371.8E3
time_array = (df_ibtracs_tc['time']-df_ibtracs_tc['time'].iloc[0])/np.timedelta64(1,'s')
# zonal motion = d(lon)/dt * cos(lat) * Radius of Earth
df_ibtracs_tc['storm_motion_u'] = np.gradient(np.radians(df_ibtracs_tc.lon.values),time_array.values)*REarth*np.cos(np.radians(df_ibtracs_tc.lat.values))
# meridional motion = d(lat)/dt * Radius of Earth
df_ibtracs_tc['storm_motion_v'] = np.gradient(np.radians(df_ibtracs_tc.lat.values),time_array.values)*REarth
df_ibtracs_tc

In [ ]:
#Extract dropsonde data for specified mission(s) of interest
#subset mission(s) of interest
ind = []
for i in range(len(flightid)):
    ind.append(np.nonzero(flightids == flightid[i])[0])
ind = np.hstack(ind)

#get the launch time for each drop
drops = ds_tcdrops[['year','month','day','hour','minute','second','latitude_initial','longitude_initial',\
                       'lat_dist_initial','lon_dist_initial','storm_name','flight_id']].isel(ndrop = ind)
df_mission = drops.to_dataframe()
df_mission['storm_name'] = df_mission['storm_name'].str.decode("utf-8")
df_mission['flight_id'] = df_mission['flight_id'].str.decode("utf-8")

#convert launch date/time information to datetime
df_mission['launchtime'] = df_mission.apply(lambda row: datetime(int(row['year']),int(row['month']),int(row['day']),\
                                        int(row['hour']),int(row['minute']),int(row['second'])) ,axis=1)

#get all drops for mission
ds_tcdrop = ds_tcdrops[['time_elapsed','longitude','latitude','pressure','height','temperature','wvmr','rh','uwind','vwind','theta_e']].isel(ndrop = ind)

#convert to dataframe and clean up
df_tcdrop = ds_tcdrop.to_dataframe().dropna(how='all').reset_index()

#calculate time of observation
df_tcdrop = pd.merge(df_tcdrop,df_mission['launchtime'],on='ndrop')
df_tcdrop['time'] = df_tcdrop['launchtime'] + pd.to_timedelta(df_tcdrop['time_elapsed'], unit='s')
df_tcdrop = df_tcdrop.drop(columns=['time_elapsed','nlevels','launchtime'])

#merge with resampled IBTracs data
df_tcdrop['time'] = df_tcdrop['time'].dt.round('s') #round time in TC-DROPS to nearest second
df_tcdrop = pd.merge(df_tcdrop,df_ibtracs_tc,on='time')

#rename TC position columns to avoid confusion
df_tcdrop = df_tcdrop.rename(columns={'lat': 'tclat', 'lon': 'tclon'})

df_tcdrop

In [ ]:
#Perform calculations to transform variables into cylindrical grid and storm-relative frame of reference
#calculate x and y distance relative to center, along with radius and azimuth
tcphi = df_tcdrop.tclat*np.pi/180
tclam = df_tcdrop.tclon*np.pi/180
phi = df_tcdrop.latitude*np.pi/180
lam = df_tcdrop.longitude*np.pi/180

#transformation variables to account for sphericity of lat/lon grid, following Ahern and Cowan (2018)
a = np.clip(np.sin(tcphi)*np.sin(phi)+np.cos(tcphi)*np.cos(phi)*np.cos(phi-tcphi),-1,1)
ctrangle = np.maximum(np.arccos(a),1E-3)
df_tcdrop['x_dist'] = ctrangle*(REarth+df_tcdrop.height)/np.sin(ctrangle)*np.cos(phi)*np.sin(lam-tclam)
df_tcdrop['y_dist'] = ctrangle*(REarth+df_tcdrop.height)/np.sin(ctrangle)*(np.cos(tcphi)*np.sin(phi)-\
                                np.sin(tcphi)*np.cos(phi)*np.cos(lam-tclam))
df_tcdrop['radius'] = (df_tcdrop.x_dist**2.0+df_tcdrop.y_dist**2.0)**0.5
theta_xy = np.arctan2(df_tcdrop.y_dist,df_tcdrop.x_dist)
df_tcdrop['azimuth'] = np.where(theta_xy>=0,theta_xy,theta_xy+2*np.pi)

In [ ]:
#Calculate storm-relative u, v, radial, and tangential wind, following Ahern and Cowan (2018)
df_tcdrop['u_wind_relative']=df_tcdrop.uwind-df_tcdrop.storm_motion_u
df_tcdrop['v_wind_relative']=df_tcdrop.vwind-df_tcdrop.storm_motion_v

eps = 1.E-3
wspd = np.sqrt(df_tcdrop.u_wind_relative**2+df_tcdrop.v_wind_relative**2)
lamprime = lam + eps*df_tcdrop.u_wind_relative/(wspd*np.cos(phi))
phiprime = phi + eps*df_tcdrop.v_wind_relative/wspd

b = np.clip(np.sin(tcphi)*np.sin(phiprime)+np.cos(tcphi)*np.cos(phiprime)*np.cos(phiprime-tcphi),-1,1)
ctrangleprime = np.maximum(np.arccos(b),1E-3)
xprime = ctrangleprime*(REarth+df_tcdrop.height)/np.sin(ctrangleprime)*np.cos(phiprime)*np.sin(lamprime-tclam)
yprime = ctrangleprime*(REarth+df_tcdrop.height)/np.sin(ctrangleprime)*(np.cos(tcphi)*np.sin(phiprime)-\
                        np.sin(tcphi)*np.cos(phiprime)*np.cos(lamprime-tclam))
alpha = np.arctan2(yprime-df_tcdrop.y_dist,xprime-df_tcdrop.x_dist)
alpha = np.where(alpha>=0,alpha,alpha+2*np.pi)

df_tcdrop['radwind_relative'] = wspd*np.cos(df_tcdrop.azimuth-alpha)
df_tcdrop['tanwind_relative'] = wspd*np.cos(df_tcdrop.azimuth-alpha+np.pi/2)

In [ ]:
#Get rid of NaNs and sort
df_tcdrop = df_tcdrop.set_index('time').groupby('ndrop', group_keys=True).apply(lambda group : group.interpolate(method='time',limit_area='inside'))
df_tcdrop = df_tcdrop.drop(columns=['ndrop'])
df_tcdrop = df_tcdrop.dropna().sort_values(['ndrop','time'],ascending=[True,False]).reset_index()

df_tcdrop

#### Examples diagnostics and visualizations

In [ ]:
#Plot dropsonde points in TC-relative frame of reference
#group into drops from low-level and high-level aircraft
pltlim = 400
df_tcdrop_release = df_tcdrop.groupby('ndrop').tail(1)
df_tcdrop_release_hi = df_tcdrop_release[df_tcdrop_release['pressure']<500] #low-level recon drops
df_tcdrop_release_lo = df_tcdrop_release[df_tcdrop_release['pressure']>=500] #high-level recon drops

#plot release points (note: only plots points within axes range)
fig = plt.figure(figsize=(8,8))
plt.plot(df_tcdrop_release_hi['x_dist']/1E3,df_tcdrop_release_hi['y_dist']/1E3,'x',color='red',ms=7,mew=1.5)
plt.plot(df_tcdrop_release_lo['x_dist']/1E3,df_tcdrop_release_lo['y_dist']/1E3,'+',color='blue',ms=8,mew=1.5)
for j in range(len(df_tcdrop_release)):
    plt.text(df_tcdrop_release['x_dist'].iloc[j]/1E3,df_tcdrop_release['y_dist'].iloc[j]/1E3,df_tcdrop_release['ndrop'].iloc[j],clip_on=True)
plt.ylim(-pltlim,pltlim)
plt.xlim(-pltlim,pltlim)

#range rings
crad = np.arange(100,401,100)
for i in range(len(crad)):
    c = plt.Circle((0,0), radius=crad[i], edgecolor='gray', facecolor='none', linestyle='--', linewidth=0.5)
    plt.gca().add_artist(c)
plt.plot([-400,400],[0,0],color='gray',linestyle='--', linewidth=0.5)
plt.plot([0,0],[-400,400],color='gray',linestyle='--', linewidth=0.5)

plt.xlabel('zonal distance from center (km)')
plt.ylabel('meridional distance from center (km)')
plt.show()

In [ ]:
#Plot skew-T and ventilation profile
#select dropsonde to plot
sid = 78 #isolate 

fig = plt.figure(figsize=(8,8))
skew = SkewT(fig, rotation=35, aspect=90)

df_sonde = df_tcdrop[(df_tcdrop['ndrop'] == sid)].sort_values(by=['pressure'])
p = df_sonde['pressure'].values * units.hPa
T = df_sonde['temperature'].values * units.degC
u = df_sonde['uwind'].values * units('m/s')
v = df_sonde['vwind'].values * units('m/s')
Td = dewpoint_from_relative_humidity(df_sonde['temperature'].values * units.degC, df_sonde['rh'].values * units.percent)
skew.plot(p, T, lw=2.5, color='tab:red')
skew.plot(p, Td,lw=2.5, color='tab:green')
skew.plot_barbs(p[::10], u[::10], v[::10], length=6)

skew.ax.set_ylim(1012, 200)
skew.ax.set_xlim(-25, 30)
skew.ax.set_xlabel('Temperature, dewpoint (C)')
skew.ax.set_ylabel('Pressure (hPa)')

skew.plot_dry_adiabats(lw=0.5)
skew.plot_moist_adiabats(lw=0.5)
skew.plot_mixing_lines(lw=0.5)

plt.show()

In [ ]:
#Make a plot of values of some variable averaged over a given layer. Use theta_e as an example.
#determine change of pressure between observations to use as weights in layer average
df_tcdrop['deltap'] = ((df_tcdrop['pressure']+df_tcdrop['pressure'].shift(1))/2) - \
                      ((df_tcdrop['pressure']+df_tcdrop['pressure'].shift(-1))/2)
df_tcdrop['deltap'] = df_tcdrop['deltap'].fillna(0)
df_tcdrop['deltap'] = df_tcdrop['deltap'].mask(df_tcdrop['deltap'] < 0, 0)

#specify layer of interest that you want to average over
df_lay = df_tcdrop.loc[(df_tcdrop['pressure'] >= 900) & (df_tcdrop['pressure'] <= 1000)].groupby('ndrop')

#calculate layer average (add variables to list that you want to average)
df_lay_avg = df_lay.apply(lambda x: pd.Series(np.average(x[['x_dist','y_dist','theta_e']], \
                            weights=x['deltap'], axis=0), ['x_dist','y_dist','theta_e'])).dropna()

#determine radius of azimuth of average locations
theta_xy = np.arctan2(df_lay_avg.y_dist,df_lay_avg.x_dist)
df_lay_avg['radius'] = (df_lay_avg.x_dist**2.0+df_lay_avg.y_dist**2.0)**0.5
df_lay_avg['azimuth'] = np.where(theta_xy>=0,theta_xy,theta_xy+2*np.pi)

In [ ]:
#make plot
pltlim = 400
fig = plt.figure(figsize=(10,8))
h = plt.scatter(df_lay_avg['x_dist']/1E3,df_lay_avg['y_dist']/1E3,c=df_lay_avg['theta_e'],s=150,edgecolors='black',cmap=plt.cm.Spectral_r,vmin=340,vmax=370)
cb = plt.colorbar(h, shrink=0.75)
cb.set_label(r'$\theta_e\;(K)$')
plt.ylim(-pltlim,pltlim)
plt.xlim(-pltlim,pltlim)

crad = np.arange(100,401,100)
for i in range(len(crad)):
    c = plt.Circle((0,0), radius=crad[i], edgecolor='gray', facecolor='none', linestyle='--', linewidth=0.5)
    plt.gca().add_artist(c)
plt.plot([-400,400],[0,0],color='gray',linestyle='--', linewidth=0.5)
plt.plot([0,0],[-400,400],color='gray',linestyle='--', linewidth=0.5)

plt.xlabel('zonal distance from center (km)')
plt.ylabel('meridional distance from center (km)')
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from pyproj import Geod
from datetime import timedelta

geod = Geod(ellps='WGS84')

def label_RI_RW(bt_df, hours_between=24, wind_col='vmax_kt', time_col='time'):
    bt = bt_df.sort_values(time_col).set_index(time_col)
    step = int(hours_between / (bt.index.to_series().diff().median().total_seconds() / 3600.0))
    bt['vmax_next'] = bt[wind_col].shift(-step)
    bt['dv'] = bt['vmax_next'] - bt[wind_col]
    RI = bt[bt['dv'] >= 30]
    RW = bt[bt['dv'] <= -30]
    return RI.index.tolist(), RW.index.tolist(), bt

def to_storm_relative(storm_lat, storm_lon, pts_df):
    azs, dists = [], []
    for lat, lon in zip(pts_df['lat'].values, pts_df['lon'].values):
        az1, az2, dist = geod.inv(storm_lon, storm_lat, lon, lat)
        azs.append((az1 + 360) % 360)
        dists.append(dist/1000.0)
    pts = pts_df.copy()
    pts['azimuth_deg'] = azs
    pts['distance_km'] = dists
    return pts

def compute_event_metrics(onset_time, bt_df, flashes_df, outer_rmin=100, outer_rmax=500, pre_hours=24, post_hours=24):
    # find storm position at onset_time (interpolate if necessary)
    storm_row = bt_df.loc[onset_time]
    s_lat, s_lon = storm_row['lat'], storm_row['lon']
    # window
    t0 = pd.to_datetime(onset_time)
    win = (flashes_df['time'] >= t0 - pd.Timedelta(hours=pre_hours)) & (flashes_df['time'] < t0 + pd.Timedelta(hours=post_hours))
    fl_win = flashes_df[win].copy()
    fl_rel = to_storm_relative(s_lat, s_lon, fl_win)
    # outer flashes pre/post
    pre_mask = (fl_rel['time'] >= t0 - pd.Timedelta(hours=pre_hours)) & (fl_rel['time'] < t0)
    post_mask = (fl_rel['time'] >= t0) & (fl_rel['time'] < t0 + pd.Timedelta(hours=post_hours))
    outer_mask = (fl_rel['distance_km'] >= outer_rmin) & (fl_rel['distance_km'] <= outer_rmax)
    outer_pre = fl_rel[pre_mask & outer_mask]
    outer_post = fl_rel[post_mask & outer_mask]
    # counts and rates
    metrics = {
        'outer_pre_count': len(outer_pre),
        'outer_post_count': len(outer_post),
        'outer_pre_rate_per_hr': len(outer_pre) / pre_hours,
        'outer_post_rate_per_hr': len(outer_post) / post_hours,
    }
    # area km^2
    import math
    area = math.pi * (outer_rmax**2 - outer_rmin**2)
    metrics['outer_pre_density_per_km2_hr'] = metrics['outer_pre_rate_per_hr'] / area
    metrics['outer_post_density_per_km2_hr'] = metrics['outer_post_rate_per_hr'] / area
    return metrics


In [ ]:
#---------------------------------------------------------------------------------------------------------------------------

In [ ]:
#--------------               WITH ENV DATA                  -----------------------------

In [ ]:
import metpy.calc as mpcalc
from metpy.plots import SkewT
from metpy.units import units

# Pick one dropsonde ID
drop_id = 12

df_skew = df_tcdrop[df_tcdrop["ndrop"] == drop_id].sort_values("pressure")

# Units
p = df_skew["pressure"].values * units.hPa
T = df_skew["temperature"].values * units.degC
rh = df_skew["rh"].values / 100.0

Td = mpcalc.dewpoint_from_relative_humidity(T, rh)
u = df_skew["uwind"].values * units("m/s")
v = df_skew["vwind"].values * units("m/s")


In [ ]:
fig = plt.figure(figsize=(6, 6))
skew = SkewT(fig, rotation=45)

skew.plot(p, T, 'r', linewidth=2)
skew.plot(p, Td, 'g', linewidth=2)
# Plot barbs every 5th level
skew.plot_barbs(p[::10], u[::10], v[::10])


skew.ax.set_ylim(1000, 100)
skew.ax.set_xlim(-30, 40)

skew.plot_dry_adiabats(alpha=0.4)
skew.plot_moist_adiabats(alpha=0.4)
skew.plot_mixing_lines(alpha=0.4)

plt.title(f"Dropsonde {drop_id} | {df_skew.time.iloc[0]}")
plt.show()


In [ ]:
theta_e = mpcalc.equivalent_potential_temperature(p, T, Td)

plt.figure(figsize=(4, 6))
plt.plot(theta_e, p)
plt.gca().invert_yaxis()
plt.xlabel("Equivalent Potential Temperature (K)")
plt.ylabel("Pressure (hPa)")
plt.title(f"θe Profile | Dropsonde {drop_id}")
plt.grid()
plt.show()


In [ ]:
# ---------------------------- QUADRANT GRAPHS 

In [ ]:
def assign_quadrant(row):
    if row.x_dist > 0 and row.y_dist > 0:
        return "NE"
    elif row.x_dist < 0 and row.y_dist > 0:
        return "NW"
    elif row.x_dist < 0 and row.y_dist < 0:
        return "SW"
    elif row.x_dist > 0 and row.y_dist < 0:
        return "SE"
    else:
        return "Center"

df_tcdrop["quadrant"] = df_tcdrop.apply(assign_quadrant, axis=1)


In [ ]:
df_env = df_tcdrop[
    (df_tcdrop.radius >= 100e3) &
    (df_tcdrop.radius <= 300e3)
]


In [ ]:
from metpy.plots import SkewT
from metpy.calc import dewpoint_from_relative_humidity
from metpy.units import units

fig = plt.figure(figsize=(10, 10))
quadrants = ["NE", "NW", "SE", "SW"]

for i, quad in enumerate(quadrants):
    df_q = df_env[df_env.quadrant == quad]

    numeric_cols = [
        "temperature",
        "rh",
        "uwind",
        "vwind",
        "theta_e"
    ]

    prof = (
        df_q
        .groupby("pressure")[numeric_cols]
        .median()
        .dropna(subset=["temperature", "rh"])
        .sort_index(ascending=False)
    )

    p = prof.index.values * units.hPa
    T = prof.temperature.values * units.degC
    Td = dewpoint_from_relative_humidity(
        T, prof.rh.values * units.percent
    )

    skew = SkewT(fig, subplot=(2, 2, i + 1), rotation=45)

    skew.plot(p, T, 'r', linewidth=2)
    skew.plot(p, Td, 'g', linewidth=2)

    skew.ax.set_ylim(1000, 100)
    skew.ax.set_xlim(-30, 40)

    skew.plot_dry_adiabats(alpha=0.3)
    skew.plot_moist_adiabats(alpha=0.3)
    skew.plot_mixing_lines(alpha=0.3)

    skew.ax.set_title(f"{quad} Quadrant", fontsize=11)

plt.suptitle("Median Environmental Skew-T by Storm Quadrant for 14th 13:00:34 → 15th 08:07:47", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
#--------------------------------                         NE CLUSTER

In [ ]:
# Define NE convective cluster
df_ne_cluster = df_env[
    (df_env.azimuth >= 0) &
    (df_env.azimuth < 90) &
    (df_env.radius >= 150) &
    (df_env.radius <= 350)
]

print("Number of dropsondes in NE cluster:",
      df_ne_cluster.ndrop.nunique())

print("Time range:")
print(df_ne_cluster.time.min(), "to", df_ne_cluster.time.max())

print("Dropsonde IDs:")
print(sorted(df_ne_cluster.ndrop.unique()))


In [ ]:
df_env.azimuth.describe()


In [ ]:
df_env.azimuth.sort_values().unique()[:10], df_env.azimuth.sort_values().unique()[-10:]


In [ ]:
df_env.quadrant.value_counts()


In [ ]:
df_ne_test = df_env[
    (df_env.x_dist > 0) &
    (df_env.y_dist > 0)
]

df_ne_test.ndrop.nunique(), df_ne_test.shape


In [ ]:
df_ne_az = df_env[
    (df_env.azimuth >= 0) &
    (df_env.azimuth < np.pi / 2)
]

df_ne_az.shape


In [ ]:
df_ne_az.shape[0], df_env[df_env.quadrant == "NE"].shape[0]


In [ ]:
df_env[df_env.quadrant == "NE"].radius.describe()


In [ ]:
df_env[(df_env.quadrant=="NE") & (df_env.radius <= 350)].shape
df_env[(df_env.quadrant=="NE") & (df_env.radius >= 100)].shape
df_env[(df_env.quadrant=="NE") & (df_env.radius.between(100,350))].shape


In [ ]:
t_ne = pd.Timestamp("2014-09-15 00:30:00")


df_env[
    (df_env.quadrant=="NE") &
    (df_env.time.between(t_sat - pd.Timedelta("2H"),
                          t_sat + pd.Timedelta("2H")))
].shape


In [ ]:
t_ne = pd.Timestamp("2014-09-15 00:30:00")


In [ ]:
df_ne_cluster = df_env[
    (df_env.quadrant=="NE") &
    (df_env.radius.between(170_000, 300_000)) &
    (df_env.time.between(
        t_ne - pd.Timedelta("1h"),
        t_ne + pd.Timedelta("1h")
    ))
]


In [ ]:
print("Number of dropsonde points:", len(df_ne_cluster))
print("Unique dropsondes:", df_ne_cluster.ndrop.nunique())
print("Time range:", df_ne_cluster.time.min(), "to", df_ne_cluster.time.max())
print("Radius range (km):",
      df_ne_cluster.radius.min()/1000,
      "to",
      df_ne_cluster.radius.max()/1000)


In [ ]:
numeric_cols = [
    "temperature",
    "rh",
    "uwind",
    "vwind",
    "theta_e"
]

prof_ne = (
    df_ne_cluster
    .groupby("pressure")[numeric_cols]
    .median()
    .dropna(subset=["temperature", "rh"])
    .sort_index(ascending=False)
)


In [ ]:
from metpy.plots import SkewT
from metpy.calc import dewpoint_from_relative_humidity
from metpy.units import units

fig = plt.figure(figsize=(6, 8))
skew = SkewT(fig, rotation=45)

p = prof_ne.index.values * units.hPa
T = prof_ne.temperature.values * units.degC
Td = dewpoint_from_relative_humidity(T, prof_ne.rh.values * units.percent)

skew.plot(p, T, 'r', linewidth=2, label="T")
skew.plot(p, Td, 'g', linewidth=2, label="Td")

skew.ax.set_ylim(1000, 100)
skew.ax.set_xlim(-30, 40)

skew.plot_dry_adiabats(alpha=0.3)
skew.plot_moist_adiabats(alpha=0.3)
skew.plot_mixing_lines(alpha=0.3)

skew.ax.set_title("NE Outer Rainband Environment (~00–01 UTC)")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(5, 7))

ax.plot(prof_ne.theta_e, prof_ne.index, linewidth=2)
ax.invert_yaxis()

ax.set_xlabel("Equivalent Potential Temperature (K)")
ax.set_ylabel("Pressure (hPa)")
ax.set_title("NE Quadrant θe Profile")

plt.show()
